# DataFrame API Deep Dive: Beyond select/filter — the Full Expression API

Most Spark beginners learn `select()`, `filter()`, and `groupBy()` and call it a day. This notebook goes
much deeper: we'll explore the **expression API** (`pyspark.sql.functions`), complex transformations
like `explode()` and `pivot()`, nested struct types, and robust null/duplicate handling.

---

## Why the Expression API Matters

Spark's DataFrame API is built on top of **Catalyst**, an extensible query optimizer. Every column
expression you write — `F.upper(col("name"))`, `F.split(col("tags"), ",")` — is compiled into an
optimized JVM bytecode plan. This means:

- **Catalyst can reorder, eliminate, and fuse expressions** across your whole pipeline
- **No Python roundtrip** — unlike Python UDFs, built-in functions execute entirely in the JVM
- **Predicate pushdown** can push filters into the expression tree automatically

The golden rule: **always prefer `pyspark.sql.functions` over Python UDFs**. If a built-in function
can do the job, use it.

---

## Notebook Roadmap

| Section | Topic |
|---------|-------|
| 1 | SparkSession setup & sample data |
| 2 | String & array expressions |
| 3 | `explode()` — flattening arrays |
| 4 | `pivot()` — reshaping long → wide |
| 5 | `struct()` — nested column types |
| 6 | Null & duplicate handling |


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, ArrayType, TimestampType
)

# Connect to the standalone cluster.
# .config("spark.sql.shuffle.partitions", "8") keeps shuffle small for local dev.
spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("DataFrame API Deep Dive")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # suppress INFO noise

# --- Sample transactions dataset ---
# Each row represents one purchase event.
# 'ts' is intentionally stored as a string to show type-casting later.
data = [
    (1,  101, "Laptop",       "Electronics", 1299.99, "2024-01-15 09:30:00"),
    (2,  102, "Coffee Maker", "Appliances",   89.50,  "2024-01-15 10:05:00"),
    (3,  101, "  headphones ","Electronics",  199.00, "2024-01-16 14:20:00"),
    (4,  103, "Desk Chair",   "Furniture",    450.00, "2024-01-17 08:00:00"),
    (5,  102, "Laptop",       "Electronics", 1299.99, "2024-01-17 11:45:00"),
    (6,  104, "Blender",      "Appliances",   59.99,  "2024-01-18 16:30:00"),
    (7,  103, "Monitor",      "Electronics",  349.00, "2024-01-18 09:10:00"),
    (8,  101, "Laptop",       "Electronics", 1299.99, "2024-01-19 13:00:00"),  # duplicate product+user
    (9,  105, None,           "Furniture",    220.00, "2024-01-19 17:55:00"),  # null product
    (10, 106, "Keyboard",     "Electronics",  None,   "2024-01-20 10:00:00"),  # null amount
]

schema = StructType([
    StructField("txn_id",   IntegerType(), False),
    StructField("user_id",  IntegerType(), False),
    StructField("product",  StringType(),  True),
    StructField("category", StringType(),  True),
    StructField("amount",   DoubleType(),  True),
    StructField("ts",       StringType(),  True),  # string on purpose
])

txn_df = spark.createDataFrame(data, schema=schema)
txn_df.printSchema()
txn_df.show(truncate=False)

In [ ]:
# ============================================================
# SECTION 2: Complex Column Expressions
# ============================================================
# All transformations below use ONLY pyspark.sql.functions —
# no Python loops, no UDFs — so Catalyst can optimise them.

expr_df = txn_df.select(
    "txn_id",

    # F.concat joins strings; F.lit() creates a literal column value.
    F.concat(F.lit("TXN-"), F.col("txn_id").cast("string")).alias("txn_code"),

    # F.upper / F.lower normalise case; useful before joins on string keys.
    F.upper(F.col("category")).alias("category_upper"),
    F.lower(F.col("category")).alias("category_lower"),

    # F.trim removes leading AND trailing whitespace.
    # Notice txn_id=3 had '  headphones ' — trim fixes it.
    F.trim(F.col("product")).alias("product_trimmed"),

    # F.regexp_replace(column, pattern, replacement)
    # Replace any non-alphanumeric characters in product with '_'.
    F.regexp_replace(F.col("product"), r"[^a-zA-Z0-9]", "_").alias("product_slug"),

    # F.split(column, pattern) returns an ArrayType column.
    # Here we split the timestamp string on ' ' to get [date, time].
    F.split(F.col("ts"), " ").alias("ts_parts"),

    # F.array_contains checks if a value exists in an array column.
    # After split, check whether the date part '2024-01-15' appears.
    F.array_contains(
        F.split(F.col("ts"), " "), "2024-01-15"
    ).alias("is_jan15"),

    # Cast the string timestamp to a proper TimestampType for date arithmetic.
    F.to_timestamp(F.col("ts"), "yyyy-MM-dd HH:mm:ss").alias("ts_proper"),
)

expr_df.show(truncate=False)
expr_df.printSchema()

In [ ]:
# ============================================================
# SECTION 3: explode() — one row per array element
# ============================================================
# explode() is the idiomatic way to "unnest" array or map columns.
# Each element in the array becomes its own row; all other columns
# are duplicated for each exploded row.

# First, build a DataFrame where each user has a LIST of purchased tags.
tagged_data = [
    (101, "Alice",   ["electronics", "tech", "premium"]),
    (102, "Bob",     ["appliances", "kitchen"]),
    (103, "Charlie", ["furniture", "home", "office"]),
    (104, "Diana",   ["appliances"]),
    (105, "Eve",     []),                              # empty array — explode emits nothing
    (106, "Frank",   None),                            # null array — explode emits nothing
]

tagged_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("name",    StringType(),  True),
    StructField("tags",    ArrayType(StringType()), True),
])

tagged_df = spark.createDataFrame(tagged_data, schema=tagged_schema)
print("=== BEFORE explode ===")
tagged_df.show(truncate=False)

# F.explode(array_col) — turns [a, b, c] into three rows
exploded_df = tagged_df.select(
    "user_id",
    "name",
    F.explode(F.col("tags")).alias("tag"),  # one row per tag
)
print("=== AFTER explode ===")
exploded_df.show(truncate=False)

# TIP: use explode_outer() to preserve rows with empty/null arrays
# (they appear as a single row with tag=null instead of being dropped).
print("=== With explode_outer() — preserves empty/null arrays ===")
tagged_df.select(
    "user_id", "name",
    F.explode_outer(F.col("tags")).alias("tag")
).show(truncate=False)

In [ ]:
# ============================================================
# SECTION 4: pivot() — reshape long format to wide format
# ============================================================
# pivot() transforms distinct values of a column into new columns.
# Pattern:  df.groupBy(row_keys).pivot(pivot_col).agg(aggFunc)
#
# Without explicit pivot values, Spark does an extra pass to discover
# distinct values — always pass them explicitly in production for speed.

# Use the transactions dataset: sum(amount) per user_id per category.
pivot_df = (
    txn_df
    .groupBy("user_id")
    .pivot(
        "category",
        ["Electronics", "Appliances", "Furniture"]  # explicit values = one job instead of two
    )
    .agg(F.round(F.sum("amount"), 2))  # one aggregation function per pivoted cell
)

# Result: each category becomes a column; null means no purchases in that category.
pivot_df.show()

# BONUS: you can pivot with multiple aggregations by aliasing them,
# but that quickly creates many columns — use sparingly.
pivot_multi_df = (
    txn_df
    .groupBy("user_id")
    .pivot("category", ["Electronics", "Appliances"])
    .agg(
        F.round(F.sum("amount"), 2).alias("total"),
        F.count("txn_id").alias("cnt")
    )
)
print("Multi-agg pivot — column names follow pattern: Category_metric")
pivot_multi_df.show()

In [ ]:
# ============================================================
# SECTION 5: struct() and nested field access
# ============================================================
# F.struct() bundles multiple columns into a single StructType column.
# Access sub-fields with dot notation: col("outer.inner").

# Pack txn_id + amount + category into a nested 'purchase' struct.
struct_df = txn_df.select(
    "user_id",
    F.struct(
        F.col("txn_id"),
        F.col("product"),
        F.col("amount"),
        F.col("category"),
    ).alias("purchase"),  # nested struct column
    F.to_timestamp(F.col("ts"), "yyyy-MM-dd HH:mm:ss").alias("event_time"),
)

print("=== Schema with nested struct ===")
struct_df.printSchema()
struct_df.show(truncate=False)

# Access sub-fields using dot notation inside F.col()
struct_df.select(
    "user_id",
    F.col("purchase.txn_id").alias("txn_id"),     # drill into struct
    F.col("purchase.product").alias("product"),
    F.col("purchase.amount").alias("amount"),
    F.col("event_time"),
).show(truncate=False)

# You can also update a single sub-field while keeping the rest intact
# using withField() — available since Spark 3.1.
updated_struct_df = struct_df.withColumn(
    "purchase",
    F.col("purchase").withField("amount", F.col("purchase.amount") * 1.1)  # apply 10% markup
)
print("=== After withField() — amount increased by 10% inside struct ===")
updated_struct_df.select("user_id", "purchase.amount").show()

## Schema Evolution and Struct Types in Real Pipelines

### Why Struct Types Matter

When you ingest semi-structured data (JSON events from Kafka, nested Parquet from APIs, BSON from
MongoDB), the payload rarely maps to a flat table. Struct columns let you **preserve the original
nesting** inside Spark while still querying it with SQL semantics.

### Schema Evolution Scenarios

```
Day 1 payload                  Day 30 payload (new field added)
─────────────────────          ─────────────────────────────────
{                              {
  "purchase": {                  "purchase": {
    "txn_id": 1,                   "txn_id": 99,
    "amount": 49.99                "amount": 79.99,
  }                                "discount_pct": 10    ← NEW
}                              }
                               }
```

**With Delta Lake** (`mergeSchema = true`) or **Parquet schema merging**, new fields inside a struct
are backfilled as `null` for older partitions — your queries keep working.

### Best Practices

1. **Define schemas explicitly** using `StructType` — never rely on schema inference in production;
   inferred schemas cause an extra Spark job and can silently mistype columns.
2. **Keep raw payloads as structs** in your bronze layer; flatten only in silver/gold.
3. **Use `from_json(col, schema)`** when parsing JSON strings — it returns a StructType column
   with full Catalyst optimization support.
4. **Avoid `col("*")` on structs** after schema changes in streaming jobs — new fields won't appear
   automatically unless you restart the stream or use schema-on-read.


In [ ]:
# ============================================================
# SECTION 6: Null & Duplicate Handling
# ============================================================
# Bad data is the norm, not the exception. Spark provides first-class
# APIs for nulls and duplicates — never filter them out blindly!

print("=== Original data with nulls ===")
txn_df.show(truncate=False)

# --- dropDuplicates ---
# Without arguments: drops rows where ALL columns are identical.
# With column list: drops rows where those specific columns are identical
# (keeps the FIRST occurrence in partition order — non-deterministic across runs!).
deduped_df = txn_df.dropDuplicates(["user_id", "product"])  # one purchase per user+product
print(f"Row count before dedup: {txn_df.count()}, after: {deduped_df.count()}")
deduped_df.show(truncate=False)

# --- fillna ---
# fillna(value) fills ALL nullable columns of the matching type.
# fillna({col: value}) fills specific columns — far safer.
filled_df = txn_df.fillna({
    "product": "UNKNOWN_PRODUCT",   # string null → sentinel value
    "amount":  0.0,                  # double null → 0 (think carefully: is 0 meaningful?)
})
print("=== After fillna ===")
filled_df.show(truncate=False)

# --- replace ---
# replace() swaps specific VALUES (not nulls) in a column.
# Useful for normalising legacy codes, typo fixes, enum harmonisation.
replaced_df = filled_df.replace(
    {"UNKNOWN_PRODUCT": "N/A"},  # value mapping dict
    subset=["product"]           # only apply to this column
)
print("=== After replace: UNKNOWN_PRODUCT → N/A ===")
replaced_df.show(truncate=False)

# --- drop ---
# drop(col_name) removes a column entirely from the DataFrame.
# Accepts multiple column names or a list.
slim_df = replaced_df.drop("ts", "category")  # remove columns we no longer need
print("=== After dropping 'ts' and 'category' ===")
slim_df.show(truncate=False)

# --- Filtering nulls explicitly ---
# Sometimes you want to KEEP only rows where a key column is not null.
non_null_amounts = txn_df.filter(F.col("amount").isNotNull())
print(f"Rows with non-null amount: {non_null_amounts.count()} out of {txn_df.count()}")

In [1]:
# Clean shutdown — releases executor resources on the cluster.
spark.stop()
print("SparkSession stopped.")

NameError: name 'spark' is not defined